In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN



os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery",
        "blocksworld_mystery_2"
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [6]:
tokenizer = initialize_tokenizer(model_id)

In [7]:
import re

def find_label_positions(label_dataset, dataset):
    label_positions = {}
    for idx in tqdm(label_dataset):
        instance = dataset[idx]
        label = label_dataset[idx]["label"]
        if label is None:
            continue
        label_positions[idx] = {}
        
        for tag, content in label_dataset[idx]["label"].items():
            if content is None:
                continue
            gen_ind = instance["generation"].find(content[:30])
            
            if gen_ind == -1:
                continue
            
            tokens = tokenize_blocksworld_generation(tokenizer, instance, instance["generation"][:gen_ind])
            
            if len(tokens[0]) < 5000:
                continue
            
            label_positions[idx][tag] = len(tokens[0])
        
        label_positions[idx]["total"] = len(tokenize_blocksworld_generation(tokenizer, instance)[0])
            
    return label_positions

In [8]:
gen_datasets = [
    load_dataset(f"dmitriihook/qwq-32b-planning-{x}")["train"] for 
    x in ["mystery-24k", "mystery-2-24k"]
]

In [9]:
label_datasets = [
    load_dataset(f"dmitriihook/blocksworld-mystery-qwq-reasoning-parts-exploration")["train"],
    load_dataset(f"dmitriihook/blocksworld-mystery-2-qwq-reasoning-parts-exploration")["train"],
]

In [10]:
label_datasets = [
    {x["index"]: x for x in ld} for ld in label_datasets
]

In [11]:
label_postions_datasets = [
    find_label_positions(ld, dataset) for ld, dataset in zip(label_datasets, gen_datasets)
]

  0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

In [12]:
phrases = [
    [
    "attack",
    "succumb",
    "overcome",
    "feast",
    "province",
    "planet",
    "harmony",
    "craves",
    "pain"
    ],
    [
        "illuminate",
        "silence",
        "distill",
        "divest",
        "essence",
        "aura",
        "nexus",
        "harmonizes",
        "pulse",
    ]
]

In [13]:
def extract_all_phrase_positions(tokens: torch.Tensor, phrase: str, cot_only: bool = True) -> list[str]:
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

In [14]:
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=compute_dtype, attn_implementation="flash_attention_2", 
                                                device_map="auto")

[2025-04-11 17:46:15,190] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [15]:
from collections import defaultdict

def collect_hidden_states(ids: list[int], dataset, layers: list[int]) -> dict[int, dict]:
    hidden_states = defaultdict(dict)
    for idx in tqdm(ids):
        row = dataset[idx]
        tokens = tokenize_blocksworld_generation(tokenizer, row)
        with torch.no_grad():
            hs = model(tokens.to(device), output_hidden_states=True).hidden_states
            for layer in layers:
                hidden_states[idx][layer] = hs[layer][0].cpu().to(torch.float16).numpy()

    return hidden_states

In [16]:
n_rows = 40
layer = 47

In [17]:
def collect_correct_ids(eval_results: dict) -> list[int]:
    clean_ids = []
    for idx in range(303):
        if eval_results[idx]["llm_correct"]:
            clean_ids.append(idx)
        if len(clean_ids) == n_rows:
            break

    return clean_ids

correct_ids =[
    collect_correct_ids(eval_results[0]),
    collect_correct_ids(eval_results[1])
]

In [18]:
collected_hidden_states = [
    collect_hidden_states(correct_ids[i], gen_datasets[i], [layer]) for i in range(2)
] 

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

In [19]:
def extract_phrase_representations_batched(ids: list[int], dataset, positions_dataset, hidden_states, layer: int, phrases: list[str], steps: list[int], cont_size: int) -> dict[str, np.ndarray]:
    
    extracted_positions = defaultdict(list)
    hs = []
    
    for idx in ids:
        row = dataset[idx]
        _hs: torch.Tensor = hidden_states[idx][layer]
        
        # if idx not in positions_dataset:
        #     continue
        #     raise ValueError(f"Index {idx} not in positions dataset")
        
        # if "final-plan-formulation-verification" not in positions_dataset[idx]:
        #     continue
        #     raise ValueError(f"Index {idx} does not have final-plan-formulation-verification")
        
        # end_pos = positions_dataset[idx]["final-plan-formulation-verification"]

        generation = row["generation"]
        text = generation.split("</think>")[0]
        # text = generation

        tokens = tokenize_blocksworld_generation(tokenizer, row, text)[0]
        end_pos = len(tokens)
        reprs = {}
        
        for phrase in phrases:
            positions = extract_all_phrase_positions(tokens, phrase)
            # print(f"Phrase '{phrase}' found {len(positions)} times")
            positions = [p for p in positions if p[0] < end_pos]
            
            extracted_positions[phrase].append(positions)
            
        hs.append(_hs)

    reprs = defaultdict(list)
    
    for phrase in phrases:
        for step in steps:
            ph_positions = extracted_positions[phrase]
            step_reprs = []
            for i, positions in enumerate(ph_positions):
                _hs = hs[i]
                positions = [p for p in positions if p[0] > step - cont_size and p[1] < step]
                if len(positions) == 0:
                    continue

                for p in positions:
                    if p[1] - p[0] > 1 and p[1] < _hs.shape[0]:
                        step_reprs.append(_hs[p[0]:p[1]].mean(axis=0))
            print(
                phrase, step, len(step_reprs),
            )
                    
            reprs[f"{phrase}_{step}"] = np.stack(step_reprs).mean(axis=0)
            
    return reprs
                    
                

In [20]:
steps_1 = list(range(2000, 6000, 200))
steps_2 = list(range(1000, 7000, 200))
steps_1 = steps_2
cont_size = 100

reprs = [
    extract_phrase_representations_batched(correct_ids[0][:40], gen_datasets[0], label_postions_datasets[0], collected_hidden_states[0], layer, phrases[0], steps_1, cont_size),
    extract_phrase_representations_batched(correct_ids[1][:40], gen_datasets[1], label_postions_datasets[1], collected_hidden_states[1], layer, phrases[1], steps_2, cont_size)
]

attack 1000 27
attack 1200 10
attack 1400 24
attack 1600 11
attack 1800 17
attack 2000 20
attack 2200 25
attack 2400 30
attack 2600 14
attack 2800 26
attack 3000 60
attack 3200 36
attack 3400 36
attack 3600 49
attack 3800 36
attack 4000 42
attack 4200 33
attack 4400 39
attack 4600 33
attack 4800 31
attack 5000 28
attack 5200 30
attack 5400 33
attack 5600 36
attack 5800 30
attack 6000 30
attack 6200 42
attack 6400 23
attack 6600 40
attack 6800 30
succumb 1000 31
succumb 1200 11
succumb 1400 12
succumb 1600 17
succumb 1800 5
succumb 2000 14
succumb 2200 22
succumb 2400 14
succumb 2600 31
succumb 2800 26
succumb 3000 11
succumb 3200 41
succumb 3400 31
succumb 3600 38
succumb 3800 47
succumb 4000 50
succumb 4200 26
succumb 4400 26
succumb 4600 22
succumb 4800 26
succumb 5000 29
succumb 5200 28
succumb 5400 11
succumb 5600 19
succumb 5800 22
succumb 6000 30
succumb 6200 23
succumb 6400 19
succumb 6600 16
succumb 6800 16
overcome 1000 31
overcome 1200 43
overcome 1400 57
overcome 1600 61
ove

In [21]:
def make_mean_reprs(reprs: dict[str, np.ndarray], phrases: list[str], steps: list[int], n_last: int=3, offset: int=1) -> dict[str, np.ndarray]:
    mean_reprs = defaultdict(list)
    for phrase in phrases:
        for step in steps[-n_last - offset:-offset]:
            name = f"{phrase}_{step}"
            if name not in reprs:
                continue
            mean_reprs[phrase].append(reprs[name])
            
            
    return {k: np.stack(v, axis=0) for k, v in mean_reprs.items()}
            

In [22]:
mean_reprs = [
    make_mean_reprs(reprs[i], phrases[i], steps_1 if i == 0 else steps_2) for i in range(2)
]

In [23]:
mean_reprs_clean = {k: np.stack(v).mean(0) for k, v in mean_reprs[0].items()}
mean_reprs_mystery = {k: np.stack(v).mean(0) for k, v in mean_reprs[1].items()}

mean_clean = np.stack([mean_reprs_clean[p] for p in phrases[0]], axis=0).mean(0)
mean_mystery = np.stack([mean_reprs_mystery[p] for p in phrases[1]], axis=0).mean(0)

In [24]:
mean_reprs_clean = {
    k: v.tolist() for k, v in mean_reprs_clean.items()
}

mean_reprs_mystery = {
    k: v.tolist() for k, v in mean_reprs_mystery.items()
}

In [25]:
with open("mean_reprs_mystery_2.json", "w") as f:
    json.dump({
        "mean_reprs": mean_reprs_mystery,
        "mean_domain": mean_mystery.tolist(),
    }, f, indent=4)

In [27]:
for idx in range(20):
    row = gen_datasets[1][idx]

    text = "\n\n".join(row["generation"].split("\n\n")[:60])

    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2]

    action_postions = {
    phrase: extract_all_phrase_positions(tokens, phrase, cot_only=False)
        for phrase in phrases[1]
    }

    from collections import OrderedDict

    def forward_hook(module, input, output):
        """Replace output with the mean representation"""
        output = output[0]
        if output.shape[1] == 1:
            return (output,)
        
        output = output[0]    
        
        for ip, ph in enumerate(phrases[1]):
            positions = action_postions[ph]
            r = mean_reprs_mystery[phrases[1][ip]] - mean_mystery
            for sp, ep in positions:
                output[sp:ep] = output[sp:ep] * 0.5 + torch.tensor(r, dtype=output.dtype, device=output.device) * 0.5
            # output[positions] = torch.tensor(r, dtype=output.dtype, device=output.device)

        return (output.unsqueeze(0),)
        

    for m in model.modules():
        m._forward_hooks = OrderedDict()
        
    model.model.layers[47].register_forward_hook(forward_hook)

    with torch.no_grad():
        new_hidden_states = model(tokens.to(device), output_hidden_states=True).hidden_states
        for l in [46, 47, 48, 49]:
            torch.save(new_hidden_states[l][0].cpu(), f"hf_hidden_states/{idx}_new_{l}.pt")
        
    for m in model.modules():
        m._forward_hooks = OrderedDict()
        
    with torch.no_grad():
        new_hidden_states_clean = model(tokens.to(device), output_hidden_states=True).hidden_states
        for l in [46, 47, 48, 49]:
            torch.save(new_hidden_states_clean[l][0].cpu(), f"hf_hidden_states/{idx}_clean_{l}.pt")
        
    print("done")

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [30]:
hs_clean = torch.load("hf_hidden_states/0_clean_48.pt")
hs = torch.load("hf_hidden_states/0_new_48.pt")

/tmp/ipykernel_491937/2325316384.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hs_clean = torch.load("hf_hidden_states/0_clean_48.pt")
/tmp/ipykernel_491937/2325316384

In [ ]:
(hs - hs_clean)[:23]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.3965,  0.0273, -0.9492,  ...,  1.3125,  1.4297,  1.2969],
        [ 2.2500, -0.5977, -0.7812,  ..., -3.9375, -2.5156, -1.6719],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.bfloat16)

: 

In [29]:
hs.shape

torch.Size([1967, 5120])

In [32]:
with torch.no_grad():
    new_hidden_states_clean = model(tokens.to(device), output_hidden_states=True).hidden_states

In [30]:
new_hidden_states[48][0]

tensor([[ 10.2500,   6.5000,  -5.9375,  ...,  -1.3281,   6.8438,   2.2969],
        [-12.4375,  -7.4688,  12.3750,  ...,  34.5000,  -5.1562,  -5.2188],
        [  2.7500,   1.6875,   2.6250,  ...,   1.0625,  -2.1250,  -0.1865],
        ...,
        [  2.0625,   1.1719,   0.6289,  ...,   0.2236,   0.1709,  -2.6250],
        [  0.8750,   0.1523,  -1.7812,  ...,  12.3750,   1.0234,  -4.4375],
        [ -0.0791,  -0.7070,  -1.7031,  ...,   3.7344,   3.5469,  -5.5938]],
       device='cuda:0', dtype=torch.bfloat16)

In [33]:
new_hidden_states_clean[48][0]

tensor([[ 10.2500,   6.5000,  -5.9375,  ...,  -1.3281,   6.8438,   2.2969],
        [-12.4375,  -7.4688,  12.3750,  ...,  34.5000,  -5.1562,  -5.2188],
        [  2.7500,   1.6875,   2.6250,  ...,   1.0625,  -2.1250,  -0.1865],
        ...,
        [  1.4219,   0.4375,   1.0078,  ...,   0.3203,   0.2422,  -2.2344],
        [  0.2246,  -0.5781,  -1.3984,  ...,  12.5000,   1.0938,  -4.0625],
        [ -0.0791,  -0.7070,  -1.7031,  ...,   3.7344,   3.5469,  -5.5938]],
       device='cuda:0', dtype=torch.bfloat16)

In [38]:
row = gen_datasets[1][idx]

text = "\n\n".join(row["generation"].split("\n\n")[:60])

tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2]

action_postions = {
phrase: extract_all_phrase_positions(tokens, phrase, cot_only=False)
    for phrase in phrases[1]
}

from collections import OrderedDict

def forward_hook(module, input, output):
    """Replace output with the mean representation"""
    output = output[0]
    if output.shape[1] == 1:
        return (output,)
    
    output = output[0]    
    
    for ip, ph in enumerate(phrases[1]):
        positions = action_postions[ph]
        r = mean_reprs_mystery[phrases[1][ip]] - mean_mystery
        for sp, ep in positions:
            output[sp:ep] = output[sp:ep] * 0.5 + torch.tensor(r, dtype=output.dtype, device=output.device) * 0.5
        # output[positions] = torch.tensor(r, dtype=output.dtype, device=output.device)

    return (output.unsqueeze(0),)
    

for m in model.modules():
    m._forward_hooks = OrderedDict()
    
model.model.layers[47].register_forward_hook(forward_hook)

In [39]:
enc = model.generate(tokens.to(device), do_sample=False, max_new_tokens=3000, temperature=None, top_p=None, top_k=None, use_cache=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
print(tokenizer.decode(enc[0]))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

   Illuminate object
   Distill object from another object
   Silence object
   Divest object from another object

I have the following restrictions on my actions:
    To perform Illuminate action, the following facts need to be true: Essence object, Aura object, Nexus.
    Once Illuminate action is performed the following facts will be true: Pulse object.
    Once Illuminate action is performed the following facts will be false: Essence object, Aura object, Nexus.
    To perform Silence action, the following facts need to be true: Pulse object.
    Once Silence action is performed the following facts will be true: Essence object, Aura object, Nexus.    
    Once Silence action is performed the following facts will be false: Pulse object.
    To perform Distill action, the following needs to be true: Essence other object, Pulse object.
    Once Distill action is performed the following will be true: Nex

: 

In [ ]:

def steered_generation(idx, model): 
    row = gen_datasets[0][idx]

    text = "\n\n".join(row["generation"].split("\n\n")[:60])

    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2]

    action_postions = {
    phrase: extract_all_phrase_positions(tokens, phrase, cot_only=False)
        for phrase in phrases[0]
    }
    
    from collections import OrderedDict

    def forward_hook(module, input, output):
        """Replace output with the mean representation"""
        output = output[0]
        if output.shape[1] == 1:
            return (output,)
        
        output = output[0]    

        print("asdasd")
        
        for ip, ph in enumerate(phrases[0]):
            positions = action_postions[ph]
            r = mean_reprs_clean[phrases[0][ip]] - mean_clean
            for sp, ep in positions:
                output[sp:ep] = output[sp:ep]*0.5 + torch.tensor(r, dtype=output.dtype, device=output.device) * 0.5
            # output[positions] = torch.tensor(r, dtype=output.dtype, device=output.device)

        return (output.unsqueeze(0),)
        

    for m in model.modules():
        m._forward_hooks = OrderedDict()
        
    model.model.layers[47].register_forward_hook(forward_hook)
    
    # model = torch.compile(
    #     model, 
    #     mode="reduce-overhead",  # Less aggressive optimization but faster compilation
    #     fullgraph=False,         # Allow handling dynamic shapes
    #     dynamic=True             # Better support for variable sequence lengths
    # )
    
    print("compiled")

    with torch.no_grad():
        enc = model.generate(tokens.to(device), do_sample=False, max_new_tokens=3000, temperature=None, top_p=None, top_k=None, use_cache=True)

    return enc

In [29]:
from tqdm.auto import trange

new_generations = []

for idx in trange(20):
    new_generations.append(steered_generation(idx, model))

  0%|          | 0/20 [00:00<?, ?it/s]

compiled
asdasd
compiled
asdasd


KeyboardInterrupt: 

In [30]:
new_generations[0]

tensor([[151644,    872,    198,  ...,  11176,     60, 151645]],
       device='cuda:0')

In [31]:
idx = 0
print(tokenizer.batch_decode(new_generations[idx], skip_special_tokens=True)[0])

user
I am playing with a set of objects. Here are the actions I can do

   Attack object
   Feast object from another object
   Succumb object
   Overcome object from another object

I have the following restrictions on my actions:
    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.
    Once Attack action is performed the following facts will be true: Pain object.
    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    To perform Succumb action, the following facts need to be true: Pain object.
    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
    Once Succumb action is performed the following facts will be false: Pain object.
    To perform Overcome action, the following needs to be true: Province other object, Pain object.
    Once Overcome action is performed the following will be true: Harmony, Province

In [72]:
for x in new_generations:
    print(len(x[0]))

9140
18195
16634
8409
18963
12799


In [70]:
print(gen_datasets[0][idx]["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right. The initial conditions are:

- Block C craves Block B (so "Object Craves other object" for C and B)
- Harmony exists (Harmony is true)
- Planet Block A, Planet Block B, Planet Block D (so all these blocks are on a planet)
- Province Block A, Province Block C, Province Block D (so these are in a province, but what about Block B? Wait, the initial conditions don't mention province for Block B. Wait, the problem says "province Block A, province Block C and province Block D." So Block B's province status isn't mentioned here. Hmm, maybe it's not in a province? Or maybe it's a typo? Wait, the problem says "province Block A, province Block C and province Block D." So Block B is not in

In [50]:
for idx in correct_ids[0]:
    row = gen_datasets[0][idx]
    print(len(tokenizer.encode(row["generation"])))

10549
15977
12525
12447
7290
9523
12675
5485
11556
5666
14733
14001
7624
13020
10515
8689
11450
12786
16248
8331
5900
10683
13880
11572
9615
15093
5704
13239
10305
12455
8115
13695
7681
13664
9597
6822
11467
17068
4993
9750


In [46]:
print(tokenizer.decode(enc[0, :20000]))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

   Attack object
   Feast object from another object
   Succumb object
   Overcome object from another object

I have the following restrictions on my actions:
    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.
    Once Attack action is performed the following facts will be true: Pain object.
    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    To perform Succumb action, the following facts need to be true: Pain object.
    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
    Once Succumb action is performed the following facts will be false: Pain object.
    To perform Overcome action, the following needs to be true: Province other object, Pain object.
    Once Overcome action is performed the following will be true: Harmo

In [46]:
enc.shape

torch.Size([1, 16778])

In [49]:
print(row["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me parse the initial conditions and the goal clearly.

**Initial Conditions:**
- Block C craves Block A (so "Object Craves other object" for C and A)
- Block D craves Block C (so D craves C)
- Harmony exists (Harmony fact is true)
- Planet Block A and Planet Block B (so both are planets)
- Province Block B and Province Block D (so both B and D are in province)

**Goal:**
- Block A craves Block C
- Block C craves Block B
- Block D craves Block A

So, I need to manipulate the actions to achieve these cravings and the other conditions. Let me recall the actions and their preconditions and effects.

The actions available are Attack, Feast, Succumb, and Overcome. Let me list their requirements and effects again to make sure I have them right.

**Attack object:*

In [53]:
len(tokenize_blocksworld_generation(tokenizer, row)[0])

20191

In [47]:
len(tokenizer.encode(row["generation"]))

19520

In [77]:
print(row["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right.

**Initial Conditions:**
- Block A craves Block C (Object Craves other object: A→C)
- Block C craves Block B (C→B)
- Block D craves Block A (D→A)
- Harmony is present (Harmony)
- Planet Block B (Planet B)
- Province Block D (Province D)

**Goal:**
- Block A craves Block D (A→D)
- Block C craves Block A (C→A)
- Block D craves Block B (D→B)

So, I need to manipulate the actions (Attack, Feast, Succumb, Overcome) to transition from the initial state to the goal state. Let me recall the actions and their preconditions and effects.

Let me list out the actions again with their preconditions and effects:

1. **Attack object:**
   - Requires: Province object, Planet object, Harmony.
  